# nb06 — Per-cohort expression distributions under the three missing-value modes (Supplementary Figure S2)

**Plan:** `article_1_revision_implementation_plan_261029.md`, WP9 §9.2: old Supplementary Figures 2, 3
and 4 (one page each, 88 per-cohort histograms of the S0 dataset under strict NA handling, KNN and
softImpute) are **merged into one new Supplementary Figure S2**. The full 88-cohort set stays in the
Zenodo deposit; the figure shows 30 representative cohorts.

**What the old figures showed** (`all_cohorts_assembly.ipynb`, cell "Visualization", 8 × 11 grid):
for each cohort, a histogram of all expression values of all its samples (log2 scale). The submitted
Results (¶[92]) count the shapes: under strict NA handling 72 of 88 cohorts bimodal and 16 unimodal;
5 unimodal under KNN and 3 under softImpute; 14 trimodal under softImpute.

**Design of the merged figure.** Three panels side by side, one per missing-value mode (no
imputation, KNN, softImpute). Inside each panel the 30 cohorts are stacked as a ridgeline (one
density curve per cohort, same cohort order in the three panels), so that the change of shape from
one mode to the next can be read along a row. The colour of each curve is the platform of the cohort
(platform palette of Figure 1).

**Inputs** (S3, read as a stream, never written to the home disk):
`FL_batch_correction/exp/S0_no_removal__{strict,knn,softimpute}__01_raw__post0.tsv.gz` (7,174 samples;
3,447 genes under strict, ~13,000 under KNN / softImpute) and the prepared S0 annotation
(`rl.load_s0_annotation`).

**Outputs**
- `output/wp9/figS2_cohort_histograms.parquet` — histogram counts per cohort × mode on a fixed grid
  (all 88 cohorts; cached: the S3 pass runs only when it is missing or `FORCE_RECOMPUTE = True`);
- `output/wp9/figS2_cohort_modality.tsv` — number of density peaks per cohort × mode (all 88);
- `output/wp9/figS2_selected_cohorts.tsv` — the 30 cohorts shown and why;
- `figures_svg/figS2_merged_distributions.{svg,pdf,png}`.

**Run:** `jupyter nbconvert --to notebook --execute --inplace --ExecutePreprocessor.kernel_name=collagen_3_11 --ExecutePreprocessor.timeout=-1 nb06_merged_distributions.ipynb`
(the first run streams ~1.4 GB of gzip from S3: about 15–30 min).

Updated 2026-11-08 (figure review, plan `figure_review_corrections_plan_261108.md`): §5b S2 for all 88 cohorts (two columns × three modes, page size; Daniil decides D3); §7 before / after distributions of the 22 selected approaches → `review_extras/approach_distributions/` (internal).


## §0 Configuration
### §0a Imports

In [ ]:
# Standard libraries, plotting, S3 streaming; revision_lib only for paths, S3 client and names
import gzip
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks

REVISION_DIR = Path.cwd()
sys.path.insert(0, str(REVISION_DIR))
import revision_lib as rl  # paths, S3 client, bucket / prefix (plan WP9)

### §0b Paths and switches

| Switch | Default | Meaning |
|---|---|---|
| `IMPUTATION_MODES` | strict, KNN, softImpute | the three missing-value modes (panel order) with their plain-word titles |
| `N_COHORTS_SHOWN` | 30 | cohorts drawn in the figure (plan WP9: "30 representative ones") |
| `BIN_EDGES` | −2 … 22, step 0.1 (log2 units) | fixed histogram grid shared by every cohort and mode; values outside are clipped to the edge bins and counted |
| `CHUNK_ROWS` | 400 | samples parsed per chunk while streaming a matrix |
| `PEAK_SMOOTHING_SD` | 3 bins (0.3 log2 units) | Gaussian smoothing of a histogram before counting peaks |
| `PEAK_MIN_PROMINENCE` | 0.05 | a peak counts when its prominence is ≥ 5 % of the highest density of that cohort |
| `FORCE_RECOMPUTE` | False | re-stream the S3 matrices even if the histogram cache exists |

In [ ]:
# Paths and switches
TABLE_DIR = rl.OUTPUT_DIR / "wp9"
FIGURE_DIR = REVISION_DIR / "figures_svg"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)
HISTOGRAM_CACHE = TABLE_DIR / "figS2_cohort_histograms.parquet"

IMPUTATION_MODES = {
    "strict": "No imputation (genes with any missing value removed)",
    "knn": "KNN imputation",
    "softimpute": "softImpute imputation",
}
N_COHORTS_SHOWN = 30
BIN_EDGES = np.round(np.arange(-2.0, 22.0 + 1e-9, 0.1), 2)
BIN_CENTRES = (BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2
CHUNK_ROWS = 400
PEAK_SMOOTHING_SD = 3
PEAK_MIN_PROMINENCE = 0.05
FORCE_RECOMPUTE = False

GLOBAL_FONT_SIZE = 10
plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["figure.dpi"] = 100
plt.rcParams.update(
    {
        "font.size": GLOBAL_FONT_SIZE,
        "axes.titlesize": GLOBAL_FONT_SIZE,
        "axes.labelsize": GLOBAL_FONT_SIZE,
        "xtick.labelsize": GLOBAL_FONT_SIZE,
        "ytick.labelsize": GLOBAL_FONT_SIZE,
        "legend.fontsize": GLOBAL_FONT_SIZE,
    }
)
sns.set_style("ticks")

### §0c Platform palette

Copied from `figures_for_article/Introductory_figures_for_article.ipynb` (cell "Canonical project
palettes"; frozen, read only). The legend shows the GEO platform identifiers (as in
Supplementary File 1) and "RNA-seq".

In [ ]:
# Platform palette (copied) and plain-word platform names for the legend
platform_palette = {
    "RNASeq": "#4B0082",
    "GPL96+97": "#FF4500",
    "GPL570": "#FF8C00",
    "GPL96": "#B22222",
    "GPL6244": "#E9967A",
    "GPL13938": "#CD5C5C",
    "GPL1708": "#008080",
    "GPL14951": "#4682B4",
    "GPL17077": "#00CED1",
    "GPL13158": "#1E90FF",
    "GPL17586": "#20B2AA",
    "GPL17047": "#5F9EA0",
    "GPL887": "#B0C4DE",
    "GPL10739": "#2E8B57",
    "GPL20188": "#6B8E23",
    "GPL23541": "#8FBC8F",
    "GPL8432": "#006400",
    "GPL16686": "#8B4513",
    "GPL26356": "#A0522D",
}
# GEO platform identifiers are kept as they are (published in Supplementary File 1);
# only the RNA-seq label is spelled out
PLATFORM_PLAIN_NAMES = {p: ("RNA-seq" if p == "RNASeq" else p) for p in platform_palette}

### §0d Helpers

- `stream_cohort_histograms` — reads one gzip matrix from S3 chunk by chunk and adds every sample's
  values to its cohort's histogram on the fixed grid (no matrix is held in memory; NaN skipped).
- `count_density_peaks` — smooths a histogram and counts the peaks above the prominence threshold.
- `save_figure` — writes SVG + PDF + PNG (200 dpi).

In [ ]:
# Helpers (each with one job)
def stream_cohort_histograms(s3_key: str, cohort_of_sample: pd.Series) -> pd.DataFrame:
    # Stream one samples x genes tsv.gz from S3; return cohorts x bins histogram counts
    body = rl.s3_client().get_object(Bucket=rl.S3_BUCKET, Key=s3_key)["Body"]
    histogram_by_cohort: dict[str, np.ndarray] = {}
    n_samples_read, n_values_clipped = 0, 0
    with gzip.GzipFile(fileobj=body) as stream:
        for chunk in pd.read_csv(stream, sep="\t", index_col=0, chunksize=CHUNK_ROWS):
            chunk_cohorts = cohort_of_sample.reindex(chunk.index)
            for cohort, sample_ids in chunk.groupby(chunk_cohorts).groups.items():
                values = chunk.loc[sample_ids].to_numpy(dtype=float).ravel()
                values = values[~np.isnan(values)]
                n_values_clipped += int(((values < BIN_EDGES[0]) | (values > BIN_EDGES[-1])).sum())
                values = np.clip(values, BIN_EDGES[0], BIN_EDGES[-1])  # out-of-grid -> edge bins
                counts, _ = np.histogram(values, bins=BIN_EDGES)
                histogram_by_cohort[cohort] = histogram_by_cohort.get(cohort, 0) + counts
            n_samples_read += len(chunk)
    print(f"  {s3_key.split('/')[-1]}: {n_samples_read:,} samples, "
          f"{n_values_clipped:,} values clipped to the grid edges")
    return pd.DataFrame(histogram_by_cohort, index=BIN_CENTRES).T


def count_density_peaks(histogram_counts: np.ndarray) -> int:
    # Number of modes: peaks of the smoothed density with prominence >= threshold x max
    smoothed = gaussian_filter1d(histogram_counts.astype(float), PEAK_SMOOTHING_SD)
    peaks, _ = find_peaks(smoothed, prominence=PEAK_MIN_PROMINENCE * smoothed.max())
    return len(peaks)


def save_figure(figure: plt.Figure, stem: str) -> None:
    # Write one panel as SVG + PDF + PNG (200 dpi) into figures_svg/
    for extension in ("svg", "pdf", "png"):
        figure.savefig(FIGURE_DIR / f"{stem}.{extension}", bbox_inches="tight", dpi=200)
    print("saved", stem, "(svg, pdf, png)")

In [ ]:
# Figma units (Daniil 2026-11-09, `daniil_review_261109.md`). Figma imports an SVG with
# 1 SVG pt = 4/3 px, and the figure frame (Figma node 1607-116664) is 756 × 1159 px for the
# 180 × 275 mm page. So every panel is drawn at (slot in Figma px) × 0.75 pt, and every text
# is 7.5 pt = 10 px in Figma. Legends and colour bars ("palettes") may stick out of the slot:
# only the plot area (axes with their tick labels and titles) is fitted to it.
import matplotlib.text as mtext
from matplotlib.transforms import Bbox

FIGMA_FRAME_PX = (756, 1159)
FIGMA_PAGE_MM = (180, 275)
FIGMA_PX_PER_PT = 4 / 3
FIGMA_PX_PER_MM = FIGMA_FRAME_PX[0] / FIGMA_PAGE_MM[0]  # 4.2 px per mm of the page
FIGMA_FONT_PX = 10
FIGMA_FONT_PT = FIGMA_FONT_PX / FIGMA_PX_PER_PT  # 7.5 pt
FIGMA_FIT_LOG = rl.OUTPUT_DIR / "wp14" / "figma_fit" / "nb06_merged_distributions.tsv"
FIGMA_FIT_LOG.parent.mkdir(parents=True, exist_ok=True)

GLOBAL_FONT_SIZE = FIGMA_FONT_PT
SMALL_FONT_SIZE = FIGMA_FONT_PT
plt.rcParams.update({key: FIGMA_FONT_PT for key in (
    "font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize", "ytick.labelsize",
    "legend.fontsize", "legend.title_fontsize", "figure.titlesize")})


def mm_figsize(width_mm: float, height_mm: float) -> tuple[float, float]:
    """Figure size in inches that lands in Figma at the slot size (page mm → Figma px)."""
    to_inches = FIGMA_PX_PER_MM / FIGMA_PX_PER_PT / 72
    return width_mm * to_inches, height_mm * to_inches


def slot_figsize(stem: str) -> tuple[float, float]:
    """Figure size in inches of the page slot of a panel file (PANEL_SLOTS_MM)."""
    return mm_figsize(*PANEL_SLOTS_MM[stem])


def batch_display_name(batch: str) -> str:
    """Plain batch name: underscores → spaces; microarrays drop the last "Unknown" (library
    preparation is never known for a microarray), RNA-seq keeps its library type."""
    parts = str(batch).split("_")
    if parts[0] != "RNASeq" and len(parts) > 2 and parts[-1] == "Unknown":
        parts = parts[:-1]
    return " ".join(parts)


def is_palette_axis(ax: plt.Axes) -> bool:
    """Colour bars and legend-only axes are palettes: Daniil moves them by hand in Figma."""
    if hasattr(ax, "_colorbar") or getattr(ax, "_figma_palette", False):
        return True
    has_data = bool(ax.lines or ax.collections or ax.images or ax.patches or ax.texts)
    return not ax.axison and not has_data


def plot_extent_px(fig: plt.Figure) -> tuple[float, float]:
    """Width and height in Figma px of the plot area (axes, tick labels, titles, texts on
    the axes, suptitle), without legends and colour bars."""
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    boxes = [ax.get_tightbbox(renderer, bbox_extra_artists=list(ax.texts))
             for ax in fig.axes if ax.get_visible() and not is_palette_axis(ax)]
    boxes += [t.get_window_extent(renderer) for t in fig.texts if t.get_text()]
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text():
        boxes.append(fig._suptitle.get_window_extent(renderer))
    boxes = [b for b in boxes if b is not None and b.width > 0]
    # a legend-only figure has no plot area: measure the whole figure
    union = Bbox.union(boxes) if boxes else fig.bbox
    px_per_display = 72 / fig.dpi * FIGMA_PX_PER_PT
    return union.width * px_per_display, union.height * px_per_display


def set_all_fonts(fig: plt.Figure, size: float) -> None:
    """Every text of the figure at one size (10 px in Figma)."""
    for ax in fig.axes:
        ax.tick_params(labelsize=size)
    for text in fig.findobj(mtext.Text):
        if text.get_text():
            text.set_fontsize(size)


def fit_to_slot(fig: plt.Figure, width_mm: float, height_mm: float | None,
                n_iterations: int = 8) -> None:
    """Resize the figure until its plot area is width_mm wide (and height_mm high; None keeps
    the current aspect ratio of the plot area). Fonts keep their size, so a few redraws are
    needed; fixed-aspect plots (squares) only shrink to fit."""
    target_w = width_mm * FIGMA_PX_PER_MM
    for _ in range(n_iterations):
        plot_w, plot_h = plot_extent_px(fig)
        target_h = plot_h * target_w / plot_w if height_mm is None else height_mm * FIGMA_PX_PER_MM
        if abs(plot_w - target_w) < 1 and abs(plot_h - target_h) < 1:
            break
        fig_w, fig_h = fig.get_size_inches()
        fig.set_size_inches(max(fig_w * target_w / plot_w, 0.5), max(fig_h * target_h / plot_h, 0.5))
    plot_w, plot_h = plot_extent_px(fig)
    target_h = FIGMA_FRAME_PX[1] if height_mm is None else height_mm * FIGMA_PX_PER_MM
    shrink = min(1.0, target_w / plot_w, target_h / plot_h)
    if shrink < 0.995:  # fixed-aspect plots grew in the other direction: scale the whole panel
        fig.set_size_inches(*(fig.get_size_inches() * shrink))


def log_fit(stem: str, fig: plt.Figure, slot_mm: tuple | None) -> None:
    """One row per saved panel: slot and plot area in Figma px (read by r97_figma_check.py)."""
    plot_w, plot_h = plot_extent_px(fig)
    row = pd.DataFrame([{
        "stem": stem,
        "slot_w_px": None if slot_mm is None else round(slot_mm[0] * FIGMA_PX_PER_MM, 1),
        "slot_h_px": None if slot_mm is None or slot_mm[1] is None else round(slot_mm[1] * FIGMA_PX_PER_MM, 1),
        "plot_w_px": round(plot_w, 1), "plot_h_px": round(plot_h, 1),
    }])
    old = pd.read_csv(FIGMA_FIT_LOG, sep="\t") if FIGMA_FIT_LOG.exists() else row.iloc[:0]
    pd.concat([old[old["stem"] != stem], row]).to_csv(FIGMA_FIT_LOG, sep="\t", index=False)


def save_figure(fig: plt.Figure, stem: str, slot_mm: tuple | None = None,
                keep_fonts: bool = False) -> None:
    """Fonts → 10 Figma px (unless keep_fonts: large clustermaps), plot area → slot (default:
    PANEL_SLOTS_MM[stem] when the notebook has it), then SVG + PDF + PNG (200 dpi)."""
    if slot_mm is None and stem in globals().get("PANEL_SLOTS_MM", {}):
        slot_mm = PANEL_SLOTS_MM[stem]
    if not keep_fonts:
        set_all_fonts(fig, FIGMA_FONT_PT)
    if slot_mm is not None:
        fit_to_slot(fig, *slot_mm)
    log_fit(stem, fig, slot_mm)
    for ext in ("svg", "pdf", "png"):
        fig.savefig(FIGURE_DIR / f"{stem}.{ext}", dpi=200, bbox_inches="tight")
    print(f"saved figures_svg/{stem}.svg/.pdf/.png")


## §1 Annotation of the 7,174 samples

The prepared S0 annotation (`prepared/S0_no_removal__strict__ann.tsv.gz`, the authoritative copy) gives
the cohort (`COHORT_LABEL`, 88 levels) and the platform (`PLATFORM_RNA`, 19 levels) of each sample.
Every cohort has one platform (checked below).

In [ ]:
# Load the annotation and build the cohort table (size, platform)
annotation_s0 = rl.load_s0_annotation("strict")
cohort_of_sample = annotation_s0["COHORT_LABEL"]
cohort_table = (
    annotation_s0.groupby("COHORT_LABEL")
    .agg(n_samples=("PLATFORM_RNA", "size"),
         platform=("PLATFORM_RNA", "first"),
         n_platforms=("PLATFORM_RNA", "nunique"))
    .sort_values("n_samples", ascending=False)
)
print("samples:", len(annotation_s0), "| cohorts:", len(cohort_table),
      "| platforms:", cohort_table["platform"].nunique())
assert len(annotation_s0) == 7174 and len(cohort_table) == 88
assert (cohort_table["n_platforms"] == 1).all()
assert set(cohort_table["platform"]) <= set(platform_palette)

## §2 Histograms per cohort and missing-value mode (S3 stream, cached)

For each of the three S0 matrices without harmonization (`01_raw`, no post-removal), every value of
every sample is added to its cohort's histogram on the fixed 0.1-log2 grid. Missing values (strict
has none; KNN / softImpute keep NaN for genes that were not imputed in a cohort) are skipped. The
result, 3 modes × 88 cohorts × 240 bins, is saved as a small parquet; later runs read it.

In [ ]:
# Stream the three matrices once; afterwards read the cache
if HISTOGRAM_CACHE.exists() and not FORCE_RECOMPUTE:
    cohort_histograms = pd.read_parquet(HISTOGRAM_CACHE)
    print("read cache", HISTOGRAM_CACHE.name)
else:
    frames = []
    for mode in IMPUTATION_MODES:
        started = time.time()
        key = f"{rl.S3_PREFIX}/exp/S0_no_removal__{mode}__01_raw__post0.tsv.gz"
        histograms = stream_cohort_histograms(key, cohort_of_sample)
        histograms.columns = [f"{c:.2f}" for c in histograms.columns]
        frames.append(histograms.assign(mode=mode).rename_axis("cohort").reset_index())
        print(f"  {mode}: {time.time() - started:.0f} s")
    cohort_histograms = pd.concat(frames, ignore_index=True)
    cohort_histograms.to_parquet(HISTOGRAM_CACHE, index=False)
    print("wrote", HISTOGRAM_CACHE.name)

print(cohort_histograms.groupby("mode")["cohort"].nunique().to_string())
assert cohort_histograms.groupby("mode")["cohort"].nunique().eq(88).all()

## §3 Number of modes per cohort (check against the submitted text)

The submitted Results (¶[92]) report the counts below; they were read by eye from the old figures.
Here the peaks of each smoothed histogram are counted automatically (thresholds in §0b), so small
differences are expected; the table is a check, not a replacement of the published numbers.

| Mode | Submitted text |
|---|---|
| No imputation | 72 bimodal, 16 unimodal (of 88) |
| KNN | 5 unimodal |
| softImpute | 3 unimodal, 14 trimodal |

In [ ]:
# Count density peaks for every cohort x mode and tabulate
bin_columns = [c for c in cohort_histograms.columns if c not in ("cohort", "mode")]
cohort_modality = cohort_histograms[["cohort", "mode"]].copy()
cohort_modality["n_modes"] = [
    count_density_peaks(row) for row in cohort_histograms[bin_columns].to_numpy()
]
cohort_modality = cohort_modality.merge(
    cohort_table[["n_samples", "platform"]], left_on="cohort", right_index=True
)
cohort_modality.to_csv(TABLE_DIR / "figS2_cohort_modality.tsv", sep="\t", index=False)
modality_summary = pd.crosstab(
    cohort_modality["mode"].map({"strict": "No imputation", "knn": "KNN",
                                 "softimpute": "softImpute"}),
    cohort_modality["n_modes"].clip(upper=4).map(
        {1: "unimodal", 2: "bimodal", 3: "trimodal", 4: "4+ modes"}),
)
print(modality_summary.to_string())

## §4 Choice of the 30 cohorts shown

Rule (fixed before plotting): (1) the largest cohort of each of the 19 platforms, so that every
platform appears once; (2) the remaining 11 places go to the largest of the other cohorts. The
cohorts are drawn grouped by platform (RNA-seq first, then microarrays by platform size) and by size
inside a platform. The list is saved with the reason for each pick.

In [ ]:
# Pick one cohort per platform, then fill to 30 by size
largest_per_platform = cohort_table.reset_index().groupby("platform").head(1)["COHORT_LABEL"]
remaining = cohort_table.index.difference(largest_per_platform, sort=False)
fill_up = cohort_table.loc[remaining].head(N_COHORTS_SHOWN - len(largest_per_platform)).index
selected_cohorts = cohort_table.loc[list(largest_per_platform) + list(fill_up)].copy()
selected_cohorts["reason"] = ["largest cohort of its platform"] * len(largest_per_platform) + [
    "next largest cohort"
] * len(fill_up)
platform_size = cohort_table.groupby("platform")["n_samples"].sum()
selected_cohorts["platform_order"] = selected_cohorts["platform"].map(
    lambda p: -1e9 if p == "RNASeq" else -platform_size[p]
)
selected_cohorts = selected_cohorts.sort_values(["platform_order", "n_samples"],
                                                ascending=[True, False])
selected_cohorts.drop(columns=["platform_order", "n_platforms"]).to_csv(
    TABLE_DIR / "figS2_selected_cohorts.tsv", sep="\t"
)
print(len(selected_cohorts), "cohorts,", int(selected_cohorts["n_samples"].sum()), "samples")
print(selected_cohorts["reason"].value_counts().to_string())
assert len(selected_cohorts) == N_COHORTS_SHOWN
assert selected_cohorts["platform"].nunique() == 19

## §5 Supplementary Figure S2

Three panels (no imputation | KNN | softImpute). Each row is one cohort: its density (histogram
counts divided by the cohort total and by the bin width, lightly smoothed for display) is drawn as a
filled ridge, scaled to the same peak height in every row so that the shape, not the cohort size, is
compared. The scaling uses the highest density above 1 log2 unit; spikes of zeros or floor values
below it (RNA-seq zeros, background-corrected arrays) are cut at 1.3 × that height. Row labels: cohort name, number of samples. Colour: platform. The number of modes
counted in §3 is printed at the right end of each ridge.

In [ ]:
# Draw the ridgeline grid: 3 panels x 30 cohorts
mode_titles = {"strict": "No imputation", "knn": "KNN imputation",
               "softimpute": "softImpute imputation"}
x_limits = (0.0, 18.0)
row_step, ridge_height = 1.0, 0.85
histogram_lookup = cohort_histograms.set_index(["mode", "cohort"])[bin_columns]
modes_lookup = cohort_modality.set_index(["mode", "cohort"])["n_modes"]

figure, axes = plt.subplots(1, 3, figsize=(11, 12.5), sharey=True)
for axis, mode in zip(axes, IMPUTATION_MODES):
    for row_index, cohort in enumerate(selected_cohorts.index):
        baseline = (len(selected_cohorts) - 1 - row_index) * row_step
        counts = histogram_lookup.loc[(mode, cohort)].to_numpy(dtype=float)
        density = gaussian_filter1d(counts, 1.0)
        # scale by the peak above 1 log2 unit, so that the spike of zeros (RNA-seq) or of
        # floor values does not flatten the rest of the curve; the spike is cut at 1.3 x
        reference_peak = density[BIN_CENTRES >= 1.0].max()
        density = np.clip(density / reference_peak, 0, 1.3) * ridge_height
        colour = platform_palette[selected_cohorts.loc[cohort, "platform"]]
        axis.fill_between(BIN_CENTRES, baseline, baseline + density, color=colour,
                          alpha=0.55, linewidth=0, zorder=len(selected_cohorts) - row_index)
        axis.plot(BIN_CENTRES, baseline + density, color=colour, linewidth=0.8,
                  zorder=len(selected_cohorts) - row_index)
        axis.text(x_limits[1] - 0.1, baseline + 0.25, str(modes_lookup.loc[(mode, cohort)]),
                  ha="right", va="bottom", fontsize=8, color="#444444")
    axis.set_title(mode_titles[mode])
    axis.set_xlim(*x_limits)
    axis.set_xlabel("Expression, log2")
    sns.despine(ax=axis, left=True)
    axis.tick_params(axis="y", left=False)

axes[0].set_yticks([(len(selected_cohorts) - 1 - i) * row_step + 0.3
                    for i in range(len(selected_cohorts))])
axes[0].set_yticklabels([f"{c} (n = {selected_cohorts.loc[c, 'n_samples']})"
                         for c in selected_cohorts.index], fontsize=9)
axes[0].set_ylim(-0.3, (len(selected_cohorts) - 1) * row_step + ridge_height + 0.4)

shown_platforms = list(dict.fromkeys(selected_cohorts["platform"]))
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=platform_palette[p], alpha=0.7)
                  for p in shown_platforms]
figure.legend(legend_handles, [PLATFORM_PLAIN_NAMES[p] for p in shown_platforms],
              loc="lower center", ncol=7, frameon=False, title="Platform",
              bbox_to_anchor=(0.5, -0.07), fontsize=9)
figure.text(0.99, 0.005, "Numbers at the right: modes counted in the smoothed density",
            ha="right", fontsize=8, color="#444444")
figure.tight_layout()
save_figure(figure, "figS2_merged_distributions", slot_mm=(180, None))
plt.show()

## §5b Supplementary Figure S2, all cohorts (review figure, decision D3)

Daniil 2026-11-08: draw S2 for **all 88 cohorts**, two large columns, each with the three
missing-value modes; he looks at it and decides whether it replaces the 30-cohort S2 (D3). Same
ridges, scaling and colours as §5; cohorts ordered as in S2 (RNA-seq first, then microarray
platforms by size, inside a platform by cohort size) and split into two halves. Drawn at page size
(180 × 275 mm, fonts 6 pt). Until Daniil decides, `PLACEMENT.md`, the review PDF and the S2 legend
keep the 30-cohort figure.


In [ ]:
# Ridgelines of all 88 cohorts: two columns of cohorts × three mode panels each, at page size.
PAGE_SIZE_MM = (180, 275)  # Figma frame "Figure 4" (node 761-28) = the 180 × 275 mm text area (D2)
SMALL_FONT_SIZE = 6
mode_short_titles = {"strict": "No imputation", "knn": "KNN", "softimpute": "softImpute"}

platform_size = cohort_table.groupby("platform")["n_samples"].sum()
all_cohorts = cohort_table.assign(
    platform_order=cohort_table["platform"].map(lambda p: -1e9 if p == "RNASeq" else -platform_size[p])
).sort_values(["platform_order", "n_samples"], ascending=[True, False])
half = int(np.ceil(len(all_cohorts) / 2))
cohort_columns = [all_cohorts.index[:half], all_cohorts.index[half:]]


def draw_ridges(axis, mode: str, cohorts: pd.Index, n_rows: int, ridge: float, line: float) -> None:
    """One mode panel: one scaled density ridge per cohort, platform colour (as in §5)."""
    for row_index, cohort in enumerate(cohorts):
        baseline = (n_rows - 1 - row_index) * row_step
        density = gaussian_filter1d(histogram_lookup.loc[(mode, cohort)].to_numpy(dtype=float), 1.0)
        density = np.clip(density / density[BIN_CENTRES >= 1.0].max(), 0, 1.3) * ridge
        colour = platform_palette[cohort_table.loc[cohort, "platform"]]
        axis.fill_between(BIN_CENTRES, baseline, baseline + density, color=colour, alpha=0.55,
                          linewidth=0, zorder=n_rows - row_index)
        axis.plot(BIN_CENTRES, baseline + density, color=colour, linewidth=line, zorder=n_rows - row_index)
    axis.set_xlim(*x_limits)
    axis.set_xticks([0, 5, 10, 15])
    axis.tick_params(axis="x", labelsize=SMALL_FONT_SIZE, length=2, pad=1)
    axis.tick_params(axis="y", left=False)
    sns.despine(ax=axis, left=True)


figure = plt.figure(figsize=(PAGE_SIZE_MM[0] / 25.4, PAGE_SIZE_MM[1] / 25.4))
outer = figure.add_gridspec(1, 2, wspace=0.55, left=0.15, right=0.99, top=0.975, bottom=0.075)
for column_index, cohorts in enumerate(cohort_columns):
    inner = outer[column_index].subgridspec(1, 3, wspace=0.08)
    axes = [figure.add_subplot(inner[0, k]) for k in range(3)]
    for k, (axis, mode) in enumerate(zip(axes, IMPUTATION_MODES)):
        draw_ridges(axis, mode, cohorts, half, ridge=0.9, line=0.4)
        axis.set_ylim(-0.3, (half - 1) * row_step + 1.3)
        axis.set_title(mode_short_titles[mode], fontsize=SMALL_FONT_SIZE, pad=2)
        axis.set_xlabel("log2" if k == 1 else "", fontsize=SMALL_FONT_SIZE, labelpad=1)
        if k:
            axis.set_yticks([])
    axes[0].set_yticks([(half - 1 - i) * row_step + 0.3 for i in range(len(cohorts))])
    axes[0].set_yticklabels([f"{c} ({cohort_table.loc[c, 'n_samples']})" for c in cohorts], fontsize=SMALL_FONT_SIZE)
shown = list(dict.fromkeys(all_cohorts["platform"]))
figure.legend([plt.Rectangle((0, 0), 1, 1, color=platform_palette[p], alpha=0.7) for p in shown],
              [PLATFORM_PLAIN_NAMES[p] for p in shown], loc="lower center", ncol=10, frameon=False,
              fontsize=SMALL_FONT_SIZE, title="Platform", title_fontsize=SMALL_FONT_SIZE,
              bbox_to_anchor=(0.55, 0.0), handlelength=1, columnspacing=0.8)
save_figure(figure, "figS2_merged_distributions_all_cohorts", slot_mm=(180, None))
plt.show()
print(f"{len(all_cohorts)} cohorts in two columns of {half} / {len(all_cohorts) - half}; labels: cohort (n samples)")


## §6 Open issues for Daniil

1. **Mode counts vs the submitted text.** The automatic count (§3) uses a smoothing / prominence
   threshold; the submitted numbers (72 / 16 / 5 / 3 / 14) were read by eye. Decide whether the text
   keeps its numbers (they describe the full 88-cohort Zenodo figures) or quotes the automatic count
   from `output/wp9/figS2_cohort_modality.tsv`.
2. **30 of 88 cohorts.** The selection rule (one per platform + the largest others) is in §4. Since
   2026-11-08 §5b also draws all 88 cohorts on one page (`figS2_merged_distributions_all_cohorts`);
   Daniil decides whether it replaces the 30-cohort S2 (D3).

## §7 Before / after distributions of the 22 selected approaches (internal, "for my interest")

Daniil 2026-11-08: per approach of the final selection (22 accepted rows of
`output/wp4/selection_after_visual_check.tsv`), the per-cohort expression distributions of the
harmonized matrix next to those of the same strategy and imputation without harmonization
(`01_raw`, no post-removal). Each matrix is streamed from S3 once and reduced to per-cohort
histograms on a wider fixed grid (−5 … 25 log2 units, step 0.1; harmonized values can leave the
raw range), cached in `output/wp9/nb06_approach_histograms.parquet` (22 harmonized + the distinct
unharmonized matrices; ≈ 30–40 min on the first run, instant afterwards).

One page per approach: one row per cohort present in the harmonized matrix; grey outline =
unharmonized, filled ridge in the platform colour = harmonized (both scaled to their own peak, as
in S2). Output `review_extras/approach_distributions/<scenario>__<approach>.pdf` and the merged
`all_22_approaches.pdf`. Not a publication figure (not in `figures_svg/`, PLACEMENT or r46).


In [ ]:
# Stream the 22 harmonized + their unharmonized matrices once (cache), then draw one page each.
from matplotlib.backends.backend_pdf import PdfPages

MAKE_APPROACH_DISTRIBUTIONS = True
APPROACH_CACHE = TABLE_DIR / "nb06_approach_histograms.parquet"
APPROACH_DIR = REVISION_DIR / "review_extras" / "approach_distributions"
APPROACH_BIN_EDGES = np.round(np.arange(-5.0, 25.0 + 1e-9, 0.1), 2)
APPROACH_BIN_CENTRES = (APPROACH_BIN_EDGES[:-1] + APPROACH_BIN_EDGES[1:]) / 2
IMPUTATION_PLAIN = {"strict": "no imputation", "knn": "KNN", "softimpute": "softImpute"}

selection = pd.read_csv(rl.OUTPUT_DIR / "wp4" / "selection_after_visual_check.tsv", sep="\t")
selection = selection[selection["visual_check"] == "accepted"].reset_index(drop=True)
assert len(selection) == 22, len(selection)
selection["harmonized_key"] = [rl.main_benchmark_matrix_key(r.strat, r.imp, r.method, bool(r.post_rm)) for r in selection.itertuples()]
selection["raw_key"] = [rl.main_benchmark_matrix_key(r.strat, r.imp, "01_raw", False) for r in selection.itertuples()]
needed_keys = list(dict.fromkeys(list(selection["harmonized_key"]) + list(selection["raw_key"])))
print(f"{len(selection)} approaches, {len(needed_keys)} distinct matrices")


def stream_histograms_on_grid(s3_key: str) -> pd.DataFrame:
    """Per-cohort histogram of one matrix on APPROACH_BIN_EDGES (chunked S3 stream, NaN skipped)."""
    body = rl.s3_client().get_object(Bucket=rl.S3_BUCKET, Key=s3_key)["Body"]
    counts_by_cohort: dict[str, np.ndarray] = {}
    with gzip.GzipFile(fileobj=body) as stream:
        for chunk in pd.read_csv(stream, sep="\t", index_col=0, chunksize=CHUNK_ROWS):
            for cohort, ids in chunk.groupby(cohort_of_sample.reindex(chunk.index)).groups.items():
                values = chunk.loc[ids].to_numpy(dtype=float).ravel()
                values = np.clip(values[~np.isnan(values)], APPROACH_BIN_EDGES[0], APPROACH_BIN_EDGES[-1])
                counts_by_cohort[cohort] = counts_by_cohort.get(cohort, 0) + np.histogram(values, bins=APPROACH_BIN_EDGES)[0]
    table = pd.DataFrame(counts_by_cohort, index=[f"{c:.2f}" for c in APPROACH_BIN_CENTRES]).T
    return table.rename_axis("cohort").reset_index().assign(s3_key=s3_key)


if MAKE_APPROACH_DISTRIBUTIONS:
    cached = pd.read_parquet(APPROACH_CACHE) if APPROACH_CACHE.exists() else pd.DataFrame(columns=["s3_key"])
    missing = [k for k in needed_keys if k not in set(cached["s3_key"])]
    for number, key in enumerate(missing, 1):
        started = time.time()
        cached = pd.concat([cached, stream_histograms_on_grid(key)], ignore_index=True)
        cached.to_parquet(APPROACH_CACHE, index=False)  # saved after every matrix: a stop loses nothing
        print(f"  {number}/{len(missing)} {key.split('/')[-1]}: {time.time() - started:.0f} s")
    approach_histograms = cached.set_index(["s3_key", "cohort"])
    approach_bins = [c for c in approach_histograms.columns]


def scaled_ridge(counts: np.ndarray) -> np.ndarray:
    """Smoothed density scaled to its peak above 1 log2 unit, cut at 1.3 (as S2)."""
    density = gaussian_filter1d(counts.astype(float), 1.0)
    reference = density[APPROACH_BIN_CENTRES >= 1.0].max() or density.max() or 1.0
    return np.clip(density / reference, 0, 1.3) * 0.85


def slug(text: str) -> str:
    return "".join(ch if ch.isalnum() else "_" for ch in text).strip("_").lower()


if MAKE_APPROACH_DISTRIBUTIONS:
    APPROACH_DIR.mkdir(parents=True, exist_ok=True)
    page_files = []
    with PdfPages(APPROACH_DIR / "all_22_approaches.pdf") as merged:
        for row in selection.itertuples():
            harmonized = approach_histograms.loc[row.harmonized_key]
            raw = approach_histograms.loc[row.raw_key]
            cohorts = [c for c in all_cohorts.index if c in harmonized.index]
            title = (f"{rl.method_display_name(row.method)}, {rl.STRATEGY_DISPLAY_NAMES[row.strat]}, "
                     f"{IMPUTATION_PLAIN[row.imp]}{', post-removal' if row.post_rm else ''} — {row.scenario}")
            figure, axis = plt.subplots(figsize=(7.1, max(3.0, 0.16 * len(cohorts) + 1.2)))
            for i, cohort in enumerate(cohorts):
                base = len(cohorts) - 1 - i
                colour = platform_palette[cohort_table.loc[cohort, "platform"]]
                if cohort in raw.index:
                    axis.plot(APPROACH_BIN_CENTRES, base + scaled_ridge(raw.loc[cohort, approach_bins].to_numpy()),
                              color="#777777", linewidth=0.6, zorder=2 * (len(cohorts) - i))
                axis.fill_between(APPROACH_BIN_CENTRES, base, base + scaled_ridge(harmonized.loc[cohort, approach_bins].to_numpy()),
                                  color=colour, alpha=0.55, linewidth=0, zorder=2 * (len(cohorts) - i) - 1)
            axis.set_yticks([len(cohorts) - 1 - i + 0.3 for i in range(len(cohorts))])
            axis.set_yticklabels([f"{c} ({cohort_table.loc[c, 'n_samples']})" for c in cohorts], fontsize=6)
            axis.set_xlim(-3, 20)
            axis.set_xlabel("Expression (log2 units of the matrix)", fontsize=7)
            axis.set_title(title, fontsize=7)
            axis.tick_params(axis="x", labelsize=6)
            axis.tick_params(axis="y", left=False)
            sns.despine(ax=axis, left=True)
            axis.legend([plt.Line2D([], [], color="#777777", linewidth=0.8), plt.Rectangle((0, 0), 1, 1, color="#999999", alpha=0.55)],
                        ["Unharmonized (same strategy and imputation)", "Harmonized (platform colour)"],
                        loc="upper right", fontsize=6, frameon=False)
            figure.tight_layout()
            page = APPROACH_DIR / f"{slug(row.scenario)}__{slug(rl.method_display_name(row.method))}_{row.strat}_{row.imp}_post{int(row.post_rm)}.pdf"
            figure.savefig(page)
            merged.savefig(figure)
            page_files.append(page.name)
            plt.close(figure)
    print(f"{len(page_files)} pages → review_extras/approach_distributions/ (+ all_22_approaches.pdf)")
